Cell 1 — Install

In [1]:
!pip install httpx selectolax pydantic tenacity aiolimiter

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 1.9/1.9 MB 12.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----------------------------------- ---- 1.8/2.0 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 8.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Cell 2 — Imports

In [2]:
import asyncio
import random
import re
import time
from urllib.parse import urljoin

import httpx
from aiolimiter import AsyncLimiter
from pydantic import BaseModel, Field
from selectolax.parser import HTMLParser

Cell 2.1   → Configuration
            ├── Target_URL
            ├── BACKEND_URL
            ├── API_PREFIX
            └── API endpoints

In [4]:
# =========================
# Configuration
# =========================

BACKEND_URL = "http://localhost:8001"
API_PREFIX = "/api/v1"

PRODUCT_ENDPOINT = f"{BACKEND_URL}{API_PREFIX}/product"

In [5]:
async def send_product_to_backend(product):
    payload = product.model_dump()

    response = await http_client.client.post(
        PRODUCT_ENDPOINT,
        json=payload,
    )

    response.raise_for_status()

    return response.json()

In [6]:
result = await send_product_to_backend(products[0])

print(result)

NameError: name 'products' is not defined

Cell 3 — Product models

In [ ]:
from typing import Any


class ProductImage(BaseModel):
    url: str
    alt: str = ""
    is_primary: bool = False
    sort_order: int = 0


class RawProduct(BaseModel):
    source: str
    source_url: str

    external_id: str | None = None

    name: str
    brand: str | None = None

    sku: str | None = None
    barcode: str | None = None
    mpn: str | None = None
    gtin: str | None = None

    price: float | None = None
    compare_at_price: float | None = None
    currency: str = "BDT"

    availability: str | None = None

    description: str = ""
    short_description: str = ""

    images: list[ProductImage] = Field(
        default_factory=list
    )

    specifications: dict[str, Any] = Field(
        default_factory=dict
    )

    attributes: dict[str, Any] = Field(
        default_factory=dict
    )

Cell 4 — HTTP client

In [ ]:
class EcommerceHttpClient:

    def __init__(
        self,
        requests_per_minute=20,
        min_delay=2,
        max_delay=4,
    ):
        self.limiter = AsyncLimiter(
            requests_per_minute,
            60,
        )

        self.min_delay = min_delay
        self.max_delay = max_delay
        self.last_request = 0

        self.client = httpx.AsyncClient(
            timeout=30,
            follow_redirects=True,
            headers={
                "User-Agent": "EcommerceImporter/1.0"
            },
        )

    async def get(self, url):

        async with self.limiter:

            elapsed = (
                time.monotonic()
                - self.last_request
            )

            delay = random.uniform(
                self.min_delay,
                self.max_delay,
            )

            if elapsed < delay:
                await asyncio.sleep(
                    delay - elapsed
                )

            response = await self.client.get(url)

            self.last_request = time.monotonic()

            response.raise_for_status()

            return response

    async def close(self):
        await self.client.aclose()

Cell 5 — Star Tech scraper

In [ ]:
class StarTechScraper:

    BASE_URL = "https://www.startech.com.bd"

    def __init__(self, client):
        self.client = client

    async def scrape_category(self, url):

        response = await self.client.get(url)

        tree = HTMLParser(response.text)

        products = []

        for item in tree.css(".p-item"):

            product = self.parse_product_card(item)

            if product:
                products.append(product)

        return products

    def parse_product_card(self, item):

        name_node = item.css_first(".p-item-name")

        if not name_node:
            return None

        link = name_node.css_first("a")

        if not link:
            return None

        name = name_node.text(strip=True)

        href = link.attributes.get("href")

        if not href:
            return None

        url = urljoin(
            self.BASE_URL,
            href,
        )

        price = None

        price_node = item.css_first(
            "meta[itemprop='price']"
        )

        if price_node:

            value = price_node.attributes.get(
                "content"
            )

            if value:
                try:
                    price = float(value)
                except ValueError:
                    pass

        images = []

        image_node = item.css_first(
            "img[itemprop='image']"
        )

        if image_node:

            image_url = (
                image_node.attributes.get("src")
                or image_node.attributes.get("data-src")
            )

            if image_url:

                images.append(
                    ProductImage(
                        url=urljoin(
                            self.BASE_URL,
                            image_url,
                        ),
                        alt=name,
                        is_primary=True,
                    )
                )

        return RawProduct(
            source="startech",
            source_url=url,
            name=name,
            price=price,
            images=images,
        )

Cell 6 — Test

In [ ]:
client = EcommerceHttpClient()

scraper = StarTechScraper(client)

products = await scraper.scrape_category(
    "https://www.startech.com.bd/air-conditioner"
)

len(products)
print(
    products[0].model_dump_json(indent=2)
)
async def scrape_category_pages(
    scraper,
    base_url,
    max_pages=5,
):
    all_products = []

    for page in range(1, max_pages + 1):

        url = (
            base_url
            if page == 1
            else f"{base_url}?page={page}"
        )

        print(f"Scraping page {page}: {url}")

        products = await scraper.scrape_category(url)

        if not products:
            break

        all_products.extend(products)

    return all_products

products = await scrape_category_pages(
    scraper,
    "https://www.startech.com.bd/air-conditioner",
    max_pages=3,
)

print(f"Total products: {len(products)}")